In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import polars as pl
import yaml

from build_hexagon_network import (
    aggregate_by_device,
    aggregate_by_trip,
    build_conncections,
    get_daily_stop_sequence_pl,
)

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [72]:
RES = 7
VARIANT = "v1"  # of stop detection

In [73]:
path = f"{config['dirs']['output']}/{config['city']}"
Path(f"{path}/hexagon_network/{VARIANT}/{RES}").mkdir(exist_ok=True, parents=True)

In [ ]:
dummy_seq = pd.DataFrame(
    {
        "device_id": [1] * 7,
        "day": [date(2026, 5, 19)] * 5 + [date(2026, 5, 20)] * 2,
        "h3_7": [1, 2, 3, 2, 3, 1, 2],
        "latitude_mean": [47.42, 47.64, 47.23, 47.87, 47.25, 47.42, 47.64],
        "longitude_mean": [18.91, 19.12, 19.09, 19.31, 19.08, 18.91, 19.12],
    }
)

get_daily_stop_sequence_pl(pl.from_pandas(dummy_seq), "h3_7")

In [ ]:
connections = build_conncections(path, RES, VARIANT)

connections.write_parquet(f"{path}/hexagon_network/{VARIANT}/{RES}/connections.parquet")

In [ ]:
network_by_trip = aggregate_by_trip(connections)
network_by_trip.write_csv(f"{path}/hexagon_network/{VARIANT}/{RES}/edges_trip.csv")

In [ ]:
network_by_unique_device = aggregate_by_device(connections)
network_by_unique_device.write_csv(
    f"{path}/hexagon_network/{VARIANT}/{RES}/edges_device.csv"
)